In [ ]:
!pip install fastapi uvicorn pyngrok transformers torch -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

Mounted at /content/drive
Drive mounted!


In [ ]:
suggestions = {
    "sqli":    "Use parameterized queries instead of building SQL strings manually.",
    "xss":     "Sanitize all user inputs before rendering them to the page.",
    "secrets": "Never hardcode API keys or passwords. Use environment variables instead.",
    "crypto":  "Avoid MD5 and SHA1. Use SHA256 or bcrypt for hashing.",
    "memory":  "Always check buffer sizes before copying data in C/C++.",
    "auth":    "Always verify user permissions before returning sensitive data.",
}

language_tips = {
    "python": [
        "Use list comprehensions instead of for loops where possible.",
        "Use f-strings for string formatting instead of .format() or %.",
        "Use 'with open()' for file handling instead of open/close.",
    ],
    "javascript": [
        "Use const and let instead of var.",
        "Use async/await instead of nested callbacks.",
        "Always use === instead of == for comparisons.",
    ],
    "java": [
        "Use try-with-resources for handling streams and connections.",
        "Use StringBuilder instead of String concatenation in loops.",
        "Prefer interfaces over abstract classes when possible.",
    ],
    "go": [
        "Always handle errors explicitly, never ignore the error return value.",
        "Use goroutines for concurrency instead of threads.",
        "Keep functions small and focused on one task.",
    ],
}

# Initialize tokenizer and model
# You might want to choose a model specifically fine-tuned for code vulnerability detection
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def analyze_code(code_text, language="python"):
    # tokenize
    inputs = tokenizer(
        code_text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=True
    )

    # get prediction
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    clean_conf = probs[0][0].item()
    vuln_conf  = probs[0][1].item()
    score      = round(clean_conf * 10, 1)

    # decide risk level
    if score >= 8:
        risk = "low"
    elif score >= 5:
        risk = "medium"
    else:
        risk = "high"

    # pick suggestions based on score
    findings = []
    if vuln_conf > 0.4:
        findings.append(suggestions["sqli"])
    if vuln_conf > 0.6:
        findings.append(suggestions["xss"])

    tips = language_tips.get(language.lower(), ["No tips available for this language yet."])

    return {
        "score":   score,
        "risk":    risk,
        "verdict": "CLEAN" if score >= 7 else "VULNERABLE",
        "clean_confidence": round(clean_conf * 100, 1),
        "vuln_confidence":  round(vuln_conf  * 100, 1),
        "findings": findings,
        "tips":     tips,
    }

print("Analyze function ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Analyze function ready!


In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

nest_asyncio.apply()

app = FastAPI(title="PolyGuard API")

# allow the PolyCode website to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# define what the request body looks like
class CodeRequest(BaseModel):
    code:     str
    language: str = "python"

# health check route
@app.get("/")
def home():
    return {"status": "PolyGuard API is running"}

# main analysis route
@app.post("/analyze")
def analyze(request: CodeRequest):
    result = analyze_code(request.code, request.language)
    return result

print("FastAPI app ready!")

FastAPI app ready!


In [54]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import threading

# Set your ngrok authtoken here. Replace "YOUR_AUTHTOKEN" with your actual token.
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
from google.colab import userdata
NGROK = userdata.get('NGROK')

# Apply nest_asyncio to allow uvicorn to run within Colab's event loop
nest_asyncio.apply()

# Terminate any existing ngrok tunnels
ngrok.kill()

# start the tunnel
public_url = ngrok.connect(8000)
print("="*50)
print("YOUR PUBLIC API URL:")
print(public_url)
print("="*50)
print("Share this URL with the PolyCode frontend team!")
print("Test it by going to:  " + str(public_url) + "/docs")

# Function to run uvicorn in a separate thread
def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start uvicorn in a separate thread
# This prevents the cell from blocking and avoids the asyncio.run() error
thread = threading.Thread(target=run_uvicorn)
thread.start()

print("Uvicorn server started in a separate thread.")
print("The Colab cell has finished execution, but the server should be running in the background.")

YOUR PUBLIC API URL:
NgrokTunnel: "https://wrongly-tray-upwind.ngrok-free.dev" -> "http://localhost:8000"
Share this URL with the PolyCode frontend team!
Test it by going to:  NgrokTunnel: "https://wrongly-tray-upwind.ngrok-free.dev" -> "http://localhost:8000"/docs
Uvicorn server started in a separate thread.
The Colab cell has finished execution, but the server should be running in the background.


In [ ]:
import requests

test_code = """
import sqlite3
user_input = "1 OR 1=1"
query = "SELECT * FROM users WHERE id = " + user_input
conn = sqlite3.connect('db.sqlite')
conn.execute(query)
"""

response = requests.post(
    "https://wrongly-tray-upwind.ngrok-free.dev/analyze",
    json={
        "code": test_code,
        "language": "python"
    }
)

print(response.json())

INFO:     34.90.127.203:0 - "POST /analyze HTTP/1.1" 200 OK
{'score': 2.8, 'risk': 'high', 'verdict': 'VULNERABLE', 'clean_confidence': 28.2, 'vuln_confidence': 71.8, 'findings': ['Use parameterized queries instead of building SQL strings manually.', 'Sanitize all user inputs before rendering them to the page.'], 'tips': ['Use list comprehensions instead of for loops where possible.', 'Use f-strings for string formatting instead of .format() or %.', "Use 'with open()' for file handling instead of open/close."]}


In [52]:
clean_code = """
import sqlite3
user_id = 5
conn = sqlite3.connect('db.sqlite')
conn.execute("SELECT * FROM users WHERE id = ?", (user_id,))
"""

response2 = requests.post(
    "https://wrongly-tray-upwind.ngrok-free.dev/analyze",
    json={
        "code": clean_code,
        "language": "python"
    }
)

try:
    print(response2.json())
except requests.exceptions.JSONDecodeError:
    print(f"Error: Could not decode JSON response. Status code: {response2.status_code}")
    print(f"Response text: {response2.text}")


Error: Could not decode JSON response. Status code: 404
Response text: <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="fon